In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.nn import GINConv, global_mean_pool
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
from sklearn.metrics import (
    roc_curve, auc, confusion_matrix,
    ConfusionMatrixDisplay, precision_score,
    recall_score, f1_score
)
import matplotlib.pyplot as plt
import pickle
import numpy as np
import random
import pandas as pd
import os

# === Reproducibility ===
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)
torch.cuda.manual_seed_all(seed)

# === Load Pickle Dataset ===
def load_pytorch_dataset(file_path):
    with open(file_path, 'rb') as f:
        dataset = pickle.load(f)
    print(f"✅ Dataset loaded from: {file_path}")
    return dataset

# === File Paths (Updated) ===
train_pickle = r"C:\Users\nidhi\Downloads\Pickle files\Hard Split\One-Hot\A+B\Train_A+B.pkl"
test_pickle  = r"C:\Users\nidhi\Downloads\Pickle files\Hard Split\One-Hot\A+B\Test_A+B.pkl"

train_dataset = load_pytorch_dataset(train_pickle)
test_dataset = load_pytorch_dataset(test_pickle)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

class SimpleGINModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, drop_out):
        super(SimpleGINModel, self).__init__()
        self.gin1 = GINConv(nn.Linear(input_dim, hidden_dim))
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.gin2 = GINConv(nn.Linear(hidden_dim, hidden_dim))
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        self.gin3 = GINConv(nn.Linear(hidden_dim, hidden_dim))
        self.bn3 = nn.BatchNorm1d(hidden_dim)
        self.gin4 = GINConv(nn.Linear(hidden_dim, hidden_dim))
        self.bn4 = nn.BatchNorm1d(hidden_dim)
        self.dropout = nn.Dropout(drop_out)
        self.fc = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = F.relu(self.bn1(self.gin1(x, edge_index)))
        x = F.relu(self.bn2(self.gin2(x, edge_index)))
        x = F.relu(self.bn3(self.gin3(x, edge_index)))
        x = F.relu(self.bn4(self.gin4(x, edge_index)))
        x = self.dropout(x)
        x = global_mean_pool(x, batch)
        return self.fc(x)

# === Training ===
def train(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct = 0, 0
    for data in loader:
        data = data.to(device)
        optimizer.zero_grad()
        data.y = data.y.long()
        out = model(data)
        loss = criterion(out, data.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * data.num_graphs
        correct += (out.argmax(dim=1) == data.y).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

# === Evaluation ===
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct = 0, 0
    all_labels, all_preds, all_probs = [], [], []
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            data.y = data.y.long()
            out = model(data)
            loss = criterion(out, data.y)
            total_loss += loss.item() * data.num_graphs
            correct += (out.argmax(dim=1) == data.y).sum().item()
            probs = F.softmax(out, dim=1)[:, 1].cpu().numpy()
            all_labels.extend(data.y.cpu().numpy())
            all_preds.extend(out.argmax(dim=1).cpu().numpy())
            all_probs.extend(probs)
    return total_loss / len(loader.dataset), correct / len(loader.dataset), \
           np.array(all_labels), np.array(all_preds), np.array(all_probs)

# === Plot Training Metrics ===
def plot_metrics(epochs, train_loss, train_acc, test_loss, test_acc, model_type):
    plt.figure(figsize=(10, 6))
    plt.plot(epochs, train_loss, color="red")
    plt.plot(epochs, test_loss, color="orange")
    plt.plot(epochs, train_acc, color="green")
    plt.plot(epochs, test_acc, color="blue")
    plt.title(f"Training Metrics ({model_type})", fontsize=14)
    plt.xlabel("Epochs", fontsize=12)
    plt.ylabel("Score", fontsize=12)
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"{model_type}_metrics_plot.png")
    plt.close()
    print(f"📈 Training metrics plot saved as {model_type}_metrics_plot.png")

# === Initialize Model ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleGINModel(input_dim=20, hidden_dim=64, output_dim=2, drop_out=0.5).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss()

# === Training Loop ===
num_epochs = 350
train_losses, train_accs, test_losses, test_accs = [], [], [], []
history = []

for epoch in range(1, num_epochs + 1):
    tr_loss, tr_acc = train(model, train_loader, optimizer, criterion, device)
    te_loss, te_acc, y_true, y_pred, y_probs = evaluate(model, test_loader, criterion, device)

    train_losses.append(tr_loss)
    train_accs.append(tr_acc)
    test_losses.append(te_loss)
    test_accs.append(te_acc)
    history.append((y_true, y_pred, y_probs))

    print(f"Epoch {epoch:03d}: Train Loss={tr_loss:.4f}, Train Acc={tr_acc:.4f}, Test Loss={te_loss:.4f}, Test Acc={te_acc:.4f}")

# === Plot Metrics ===
plot_metrics(range(1, num_epochs + 1),
             train_losses, train_accs,
             test_losses, test_accs,
             "A+B_64OneHOT_NoWD")

# === Save Metrics to Excel ===
excel_data = {
    "Epoch": list(range(1, num_epochs + 1)),
    "Train Accuracy": [round(acc * 100, 2) for acc in train_accs],
    "Train Loss": [round(loss, 2) for loss in train_losses],
    "Test Accuracy": [round(acc * 100, 2) for acc in test_accs],
    "Test Loss": [round(loss, 2) for loss in test_losses],
}

df_metrics = pd.DataFrame(excel_data)
excel_filename = "A+B_64OneHOT_NoWD.xlsx"

try:
    df_metrics.to_excel(excel_filename, index=False)
    print(f"📊 Epoch-wise metrics saved to '{excel_filename}'")
    print("📂 Saved in:", os.path.abspath(excel_filename))
except Exception as e:
    print("❌ Failed to save Excel file:", str(e))

✅ Dataset loaded from: C:\Users\nidhi\Downloads\Pickle files\Hard Split\One-Hot\A+B\Train_A+B.pkl
✅ Dataset loaded from: C:\Users\nidhi\Downloads\Pickle files\Hard Split\One-Hot\A+B\Test_A+B.pkl
Epoch 001: Train Loss=0.6935, Train Acc=0.5402, Test Loss=0.6740, Test Acc=0.6122
Epoch 002: Train Loss=0.6617, Train Acc=0.6526, Test Loss=0.6446, Test Acc=0.7296
Epoch 003: Train Loss=0.6369, Train Acc=0.7203, Test Loss=0.6335, Test Acc=0.7194
Epoch 004: Train Loss=0.6124, Train Acc=0.7688, Test Loss=0.6197, Test Acc=0.7449
Epoch 005: Train Loss=0.5944, Train Acc=0.7854, Test Loss=0.6057, Test Acc=0.7449
Epoch 006: Train Loss=0.5750, Train Acc=0.7905, Test Loss=0.5911, Test Acc=0.7449
Epoch 007: Train Loss=0.5605, Train Acc=0.7867, Test Loss=0.5836, Test Acc=0.7449
Epoch 008: Train Loss=0.5453, Train Acc=0.7803, Test Loss=0.5664, Test Acc=0.7704
Epoch 009: Train Loss=0.5302, Train Acc=0.7944, Test Loss=0.5622, Test Acc=0.7449
Epoch 010: Train Loss=0.5169, Train Acc=0.7969, Test Loss=0.5518, T

C:\Users\nidhi\AppData\Local\Temp\ipykernel_23492\3193627808.py:113: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend()


📈 Training metrics plot saved as A+B_64OneHOT_NoWD_metrics_plot.png
📊 Epoch-wise metrics saved to 'A+B_64OneHOT_NoWD.xlsx'
📂 Saved in: C:\Users\nidhi\Nidhi Programs\NEW MODELS\Hard SPLIT\ONE - HOT\No WEIGHT DECAY + Drop out\Batch size 64\A+B_64OneHOT_NoWD.xlsx
